# 🌱 Soil-Plant Yield Prediction: Ensemble Analysis
### Professional Panel Presentation - Machine Learning Pipeline

This notebook demonstrates an advanced multi-model ensemble approach to predicting agricultural yield loss based on historical environmental stressors stored in our Firebase database.

## 1. Environment Setup
Connecting to the Realtime Database and importing high-performance regression models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import firebase_admin
from firebase_admin import credentials, db
import os

# Premium Styling Configuration
sns.set_theme(style="whitegrid", palette="mako")
plt.rcParams['figure.figsize'] = [12, 7]
plt.rcParams['font.size'] = 12

# ─── DATABASE INITIALIZATION ───
DATABASE_URL = "https://soilplant-fe521-default-rtdb.asia-southeast1.firebasedatabase.app/"
CRED_PATH = 'serviceAccountKey.json'

if not firebase_admin._apps:
    if os.path.exists(CRED_PATH):
        cred = credentials.Certificate(CRED_PATH)
        firebase_admin.initialize_app(cred, {'databaseURL': DATABASE_URL})
        print("✅ Firebase Admin SDK connected.")
    else:
        print("❌ Error: 'serviceAccountKey.json' not found. Please ensure it is in the working directory.")

def load_live_training_data():
    print("📡 Fetching historical sensor records from Realtime Database...")
    try:
        ref = db.reference('sensorHistory')
        data = ref.get()
        
        if not data:
            print("⚠️ No data found in database! Ensure the ESP32 and ML Bridge are running.")
            return pd.DataFrame()
            
        # Transform dictionary of logs into DataFrame
        df = pd.DataFrame.from_dict(data, orient='index')
        
        # Maintain professional column structure
        cols = ['temperature', 'humidity', 'soilMoisture', 'soilPH', 'chlorophyll', 'turbidity', 'yieldLoss']
        df = df[cols].dropna()
        
        print(f"✅ Successfully synchronized {len(df)} samples for analysis.")
        return df
    except Exception as e:
        print(f"❌ Extraction failed: {e}")
        return pd.DataFrame()

# Load for Presentation
df = load_live_training_data()
if not df.empty:
    display(df.tail())
else:
    print("⚠️ Displaying empty dataframe. Training steps below will require data records.")

## 2. Exploratory Data Analysis
Visualizing the feature correlations to understand key yield drivers.

In [ ]:
if not df.empty:
    plt.figure(figsize=(10, 8))
    mask = np.triu(np.ones_like(df.corr(), dtype=bool))
    sns.heatmap(df.corr(), mask=mask, annot=True, cmap='coolwarm', fmt=".2f", square=True, linewidths=.5)
    plt.title('Inter-Feature Correlation Matrix', fontsize=15)
    plt.show()
else:
    print("No data available for heatmap.")

## 3. Data Preprocessing & Multi-Model Selection
Comparing multiple architectural approaches including Trees, Boosting, and Ensembles.

In [ ]:
if len(df) < 10:
    print("⚠️ Insufficient data for robust training (need >10 samples). Please run the simulation longer.")
else:
    # Define Features (X) and Target (y)
    features = ['temperature', 'humidity', 'soilMoisture', 'soilPH', 'chlorophyll', 'turbidity']
    X = df[features]
    y = df['yieldLoss']

    # Train/Test Split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Feature Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 1. Initialize Individual Models
    models = {
        "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
        "Random Forest": RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42),
        "SVR": SVR(kernel='rbf', C=100, gamma=0.1)
    }

    # 2. Create Ensemble (Voting Regressor)
    ensemble = VotingRegressor(estimators=[
        ('rf', models["Random Forest"]),
        ('gb', models["Gradient Boosting"]),
        ('dt', models["Decision Tree"])
    ])
    models["Voting Ensemble"] = ensemble

    # Train and Collect Metrics
    results = {}
    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        results[name] = {"R2": r2, "MAE": mae, "RMSE": rmse}
        print(f"✅ {name:20} trained and evaluated.")

    # Best Performing Model for final analysis
    best_model_name = "Voting Ensemble"
    final_model = models[best_model_name]
    print(f"\n🏆 Default Presentation Model: {best_model_name}")

## 4. Performance Comparison Visualization
Evaluating which model provides the highest predictive reliability (R² Score).

In [ ]:
if 'results' in locals():
    res_df = pd.DataFrame(results).T.sort_values(by="R2", ascending=False)
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=res_df.index, y=res_df["R2"] * 100, palette="mako", hue=res_df.index, legend=False)
    plt.title("Predictive Accuracy (R²) Comparison (Live DB Data)", fontsize=15, fontweight='bold')
    plt.ylabel("Accuracy (%)")
    plt.ylim(0, 110)

    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2f}%', 
                    (p.get_x() + p.get_width() / 2., p.get_height()), 
                    ha = 'center', va = 'center', 
                    xytext = (0, 9), 
                    textcoords = 'offset points', 
                    fontsize=11, fontweight='bold')
    plt.show()
    display(res_df)
else:
    print("Results not available.")

## 5. Feature Importance & Stressor Analysis
Identifying the primary drivers of yield loss based on real database records.

In [ ]:
if 'final_model' in locals() and hasattr(final_model, 'feature_importances_'):
    importances = pd.Series(final_model.feature_importances_, index=features).sort_values()
    importances.plot(kind='barh', color='darkcyan')
    plt.title('Sensor Importance in Yield Prediction (DB Data)')
    plt.xlabel('Importance Weight')
    plt.show()
elif 'final_model' in locals():
    # For Voting Regressor without pre-computed importance, average the rf/gb/dt versions
    rf_imp = models['Random Forest'].feature_importances_
    gb_imp = models['Gradient Boosting'].feature_importances_
    avg_imp = (rf_imp + gb_imp) / 2
    pd.Series(avg_imp, index=features).sort_values().plot(kind='barh', color='darkcyan')
    plt.title('Estimated Sensor Importance (Ensemble Average)')
    plt.show()
else:
    print("Model importance not available.")

## 6. Reliability Check: Predicted vs Actual
Ensuring the model tracks with the real stress patterns logged in Firebase.

In [ ]:
if 'y_test' in locals():
    predictions = final_model.predict(X_test_scaled)
    sns.jointplot(x=y_test, y=predictions, kind='reg', color='indigo')
    plt.suptitle("Prediction Reliability Analysis", y=1.02)
    plt.show()
else:
    print("Test data not available.")